# 15-phase16a-rigl-set-regrow

**neuron Phase 16a** — paradigm 의 dynamic phase 두 번째 단계. Phase 15 의 *static prune* 을 확장하여 **iterative prune + regrow** 로 학습 중 topology 가 진화하는 DST (Dynamic Sparse Training) 구현.

핵심 가설:
1. **RigL ≥ static prune?** — 같은 50% sparsity 에서 RigL DST 가 단순 prune 보다 낮은 loss?
2. **RigL ≥ SET?** — gradient-guided regrow 가 random regrow 보다 우수?
3. **constant sparsity 유지** — 각 DST cycle 후 sparsity 일정?
4. **all-finite** — DST cycle 도중 학습 안정성?

설계: full graph block (`hybrid_around_one_around_one`, Phase 14 최저 loss 구조) 위에서 **4 mode × 2 seed = 8 run**.
- `dense`: prune 없음 (baseline)
- `static_50`: 50% prune at step 750, regrow 없음 (Phase 15)
- `dst_set_50`: 50% prune at step 750 + random regrow (SET) every 50 steps, swap 10%
- `dst_rigl_50`: 50% prune at step 750 + gradient-based regrow (RigL) every 50 steps, swap 10%
- dst_end_step = 1300 (마지막 200 step 은 stabilize)

데이터: TinyShakespeare (char-LM, block_size=64)
시드: [42, 123]
작성일: 2026-05-27
연관: Issue [#76](https://github.com/EinSofINTEREST/GraphLM/issues/76) (Phase 16 main [#75](https://github.com/EinSofINTEREST/GraphLM/issues/75)) / Phase 15 PR [#72](https://github.com/EinSofINTEREST/GraphLM/pull/72)

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridTransformerTrainConfig,
    train_hybrid_transformer_lm,
)
from graphlm.utils import safe_perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")

## 1. Config + 데이터

In [ ]:
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

# Phase 15 와 동일 hyperparameter (공정 비교)
HIDDEN_DIM = 128
N_HEADS = 4
FFN_DIM = 256
N_LAYERS = 4
GROUP_SIZE = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 1500
PRUNE_AT_STEP = MAX_STEPS // 2  # 750
DST_PERIOD = 50  # 매 50 step 마다 DST cycle
DST_SWAP_FRACTION = 0.1  # alive 의 10% swap
DST_END_STEP = 1300  # 마지막 200 step stabilize
SEEDS = [42, 123]
ARCH = "hybrid_around_one_around_one"

# 4 mode 정의
MODES = [
    {"name": "dense", "prune_fraction": 0.0, "prune_at_step": None, "regrow_method": None},
    {
        "name": "static_50",
        "prune_fraction": 0.5,
        "prune_at_step": PRUNE_AT_STEP,
        "regrow_method": None,
    },
    {
        "name": "dst_set_50",
        "prune_fraction": 0.5,
        "prune_at_step": PRUNE_AT_STEP,
        "regrow_method": "random",
    },
    {
        "name": "dst_rigl_50",
        "prune_fraction": 0.5,
        "prune_at_step": PRUNE_AT_STEP,
        "regrow_method": "rigl",
    },
]
print(
    f"\nDST cycle: period={DST_PERIOD}, swap_fraction={DST_SWAP_FRACTION}, end_step={DST_END_STEP}"
)

## 2. Sweep 실행 (4 mode × 2 seed = 8 run)

In [ ]:
results = {}
for mode in MODES:
    for seed in SEEDS:
        key = (mode["name"], seed)
        print(f"\n== mode={mode['name']} seed={seed} ==")
        cfg = HybridTransformerTrainConfig(
            dataset=dataset,
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=FFN_DIM,
            n_layers=N_LAYERS,
            group_size=GROUP_SIZE,
            arch=ARCH,
            use_full_graph=True,
            block_size=BLOCK_SIZE,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            prune_at_step=mode["prune_at_step"],
            prune_fraction=mode["prune_fraction"],
            regrow_method=mode["regrow_method"],
            dst_period=DST_PERIOD if mode["regrow_method"] else None,
            dst_swap_fraction=DST_SWAP_FRACTION,
            dst_end_step=DST_END_STEP if mode["regrow_method"] else None,
            seed=seed,
            device=device,
        )
        out = train_hybrid_transformer_lm(cfg)
        results[key] = out
        n_cycles = len(out["dst_cycles"])
        print(
            f"  final_loss = {out['final_loss']:.4f}  (ppl = {safe_perplexity(out['final_loss']):.2f})"
            f"  sparsity = {out['final_sparsity']:.3f}  dst_cycles = {n_cycles}"
        )

## 3. 결과 표 + 자동 verdict

In [ ]:
print(
    f"{'mode':>12s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s} "
    f"{'sparsity':>10s} {'dst_cycles':>12s}"
)
print("-" * 75)
for (name, seed), out in results.items():
    fl = out["final_loss"]
    print(
        f"{name:>12s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f} "
        f"{out['final_sparsity']:>10.3f} {len(out['dst_cycles']):>12d}"
    )

# mode 별 평균
print("\n== Mode summary (mean ± σ across seeds) ==")
summary = {}
for mode in MODES:
    name = mode["name"]
    vals = [results[(name, s)]["final_loss"] for s in SEEDS]
    sparsities = [results[(name, s)]["final_sparsity"] for s in SEEDS]
    m = statistics.mean(vals)
    sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
    summary[name] = (m, sd, statistics.mean(sparsities))
    print(
        f"  {name:>12s} {m:.4f} ± {sd:.4f}  (ppl ≈ {safe_perplexity(m):.2f})  "
        f"sparsity={statistics.mean(sparsities):.3f}"
    )

# 자동 verdict
print("\n== Verdict ==")
dense_loss = summary["dense"][0]
static_loss = summary["static_50"][0]
set_loss = summary["dst_set_50"][0]
rigl_loss = summary["dst_rigl_50"][0]

# 1. constant sparsity — static 은 final, DST 는 모든 cycle 별 sparsity 검증
# (CodeRabbit #3308022917 — final 만 보면 cycle 중간 drift 미감지)
static_ok = abs(summary["static_50"][2] - 0.5) < 0.02
dst_cycles_ok = all(
    abs(c["sparsity_after"] - 0.5) < 0.02
    for mode_name in ("dst_set_50", "dst_rigl_50")
    for seed in SEEDS
    for c in results[(mode_name, seed)]["dst_cycles"]
)
sp_ok = static_ok and dst_cycles_ok
verdict_1 = "PASS" if sp_ok else "FAIL"
print(
    f"1. constant sparsity (static final + 모든 DST cycle, target 0.5 ±0.02): "
    f"{sp_ok}  [{verdict_1}]"
)

# 2. RigL ≥ static — DST RigL 이 static prune 보다 ≤ (낮거나 같음)
diff_rigl_static = rigl_loss - static_loss
verdict_2 = "PASS" if diff_rigl_static <= 0.02 else "FAIL"
print(f"2. RigL ≤ static + 0.02: diff = {diff_rigl_static:+.4f}  [{verdict_2}]")

# 3. RigL ≥ SET — gradient-guided 가 random 보다 ≤
diff_rigl_set = rigl_loss - set_loss
verdict_3 = "PASS" if diff_rigl_set <= 0.02 else "FAIL"
print(f"3. RigL ≤ SET + 0.02: diff = {diff_rigl_set:+.4f}  [{verdict_3}]")

# 4. all-finite
all_finite = all(math.isfinite(out["final_loss"]) for out in results.values())
verdict_4 = "PASS" if all_finite else "FAIL"
print(f"4. all-finite (DST stability): {all_finite}  [{verdict_4}]")

# DST cycle 정보 (RigL seed=42 기준 샘플)
rigl_42_cycles = results[("dst_rigl_50", 42)]["dst_cycles"]
print(f"\n== RigL seed=42 의 DST cycles ({len(rigl_42_cycles)}건) ==")
for c in rigl_42_cycles[:3]:
    print(f"  step={c['step']} swap={c['total_swap']} sparsity_after={c['sparsity_after']:.3f}")
if len(rigl_42_cycles) > 6:
    print(f"  ... (중략 {len(rigl_42_cycles) - 6}건) ...")
for c in rigl_42_cycles[-3:]:
    print(f"  step={c['step']} swap={c['total_swap']} sparsity_after={c['sparsity_after']:.3f}")

## 4. Loss curve 시각화 — prune + DST cycle 표시

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 6))
colors = {
    "dense": "tab:gray",
    "static_50": "tab:red",
    "dst_set_50": "tab:orange",
    "dst_rigl_50": "tab:blue",
}
window = 50

for mode in MODES:
    name = mode["name"]
    losses_per_seed = [results[(name, s)]["losses"] for s in SEEDS]
    smoothed = []
    for losses in losses_per_seed:
        smoothed.append(
            [
                sum(losses[max(0, i - window + 1) : i + 1]) / min(i + 1, window)
                for i in range(len(losses))
            ]
        )
    arr = torch.tensor(smoothed)
    mean = arr.mean(dim=0)
    std = arr.std(dim=0)
    steps = list(range(len(mean)))
    ax.plot(steps, mean, label=name, color=colors[name], linewidth=1.5)
    ax.fill_between(steps, mean - std, mean + std, color=colors[name], alpha=0.12)

# prune step + DST end 수직선
ax.axvline(PRUNE_AT_STEP, color="black", linestyle=":", alpha=0.5, label=f"prune @ {PRUNE_AT_STEP}")
ax.axvline(DST_END_STEP, color="gray", linestyle=":", alpha=0.5, label=f"DST end @ {DST_END_STEP}")

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title("Phase 16a — DST (RigL vs SET vs static): all 50% sparsity (mean ± σ over 2 seeds)")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase16a")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. DST cycle sparsity 추적 (RigL)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
for seed in SEEDS:
    cycles = results[("dst_rigl_50", seed)]["dst_cycles"]
    steps = [c["step"] for c in cycles]
    sparsities = [c["sparsity_after"] for c in cycles]
    ax.plot(steps, sparsities, marker="o", label=f"RigL seed={seed}", linewidth=1.5)

ax.axhline(0.5, color="black", linestyle="--", alpha=0.5, label="target sparsity 0.5")
ax.set_xlabel("step")
ax.set_ylabel("effective sparsity (mask=0 비율)")
ax.set_title("Phase 16a — DST cycle 후 sparsity (constant 유지 검증)")
ax.legend()
ax.grid(alpha=0.3)
ax.set_ylim(0.45, 0.55)
plt.tight_layout()
fig.savefig(out_dir / "sparsity_trace.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'sparsity_trace.png'}")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

- RigL vs SET vs static 의 final loss ranking?
- DST 가 static prune 보다 의미 있는 개선?
- constant sparsity 정확히 유지되는지?

**Phase 16b 진입 시 가설**:
- 16a 의 *constant sparsity DST* 가 paradigm 안에서 작동 입증 → 16b 의 *capacity expansion (Net2Net)* 도 비슷한 통합 메커니즘 (edge_mask) 위에서 작동 가능?
- 16a + 16b 결합 → *grow + shrink 동시 dynamic*